# EoMT trên BTXRD: phân đoạn u xương và phân loại u lành / u ác

Notebook này huấn luyện **EoMT (Encoder-only Mask Transformer, CVPR 2025)** của dự án trên bộ ảnh X-quang u xương **BTXRD**, theo đúng quy trình của repo: `LightningDataModule` trong `datasets/`, `LightningModule` trong `training/`, file cấu hình YAML trong `configs/`, và huấn luyện bằng `python main.py fit -c <config>`.

**Cách đặt bài toán:** instance segmentation với 2 lớp, `0 = benign` (u lành) và `1 = malignant` (u ác).
- Mỗi polygon khối u trong file LabelMe là một mask; nhãn của mask là loại lành/ác của ảnh (lấy từ `split.csv`).
- Mỗi query của EoMT dự đoán **một mask** và **một lớp**, nên cùng một mô hình vừa phân đoạn vùng u vừa phân loại lành/ác.
- Kết quả theo ảnh: vùng u = hợp các mask có điểm tin cậy u > 0.5; nhãn lành/ác = lớp của query tin cậy nhất.

**Chỉ số đánh giá**
- Phân đoạn: **Dice**, **IoU** của vùng u, cùng mask **AP** (COCO-style) như dự án gốc.
- Phân loại (malignant là lớp dương): Accuracy, Balanced accuracy, **AUROC**, F1, Sensitivity, Specificity.
- Chọn checkpoint tốt nhất theo `val_score = (Dice + AUROC) / 2`.

**Thiết lập Kaggle**
1. *Settings → Accelerator*: **GPU T4 x2** (khuyến nghị; P100 không tương thích với các bản PyTorch mới của Kaggle).
2. *Settings → Internet*: **On** (để clone repo, cài thư viện, tải trọng số DINOv2 / EoMT COCO).
3. *Add Input*: dataset BTXRD có cấu trúc như bên dưới (notebook tự tìm thư mục chứa `split.csv`):
```
BTXRD/
├── split.csv                  # image_id,split,type (benign|malignant)
├── train/{images,Annotations}/
├── val/{images,Annotations}/
└── test/{images,Annotations}/
```
   Bố cục gốc phẳng (`images/`, `Annotations/` + `split.csv` có cột `split`) cũng được hỗ trợ.
4. (Tùy chọn) *Add-ons → Secrets*: thêm `WANDB_API_KEY` để log lên Weights & Biases như dự án gốc. Nếu không có, notebook chỉ log ra CSV.

## 1. Cấu hình

In [ ]:
import os
import sys
import csv
import json
import glob
import math
import shutil
import subprocess
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()
WORK_DIR = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

# --- Source code ---
REPO_URL = "https://github.com/67htb5vn/eomt.git"  # fork of tue-mps/eomt
REPO_DIR = WORK_DIR / "eomt" if ON_KAGGLE else Path.cwd()

# --- Data ---
DATA_ROOT = None  # None = auto-detect the folder holding split.csv in /kaggle/input
CACHE_DIR = WORK_DIR / "btxrd_cache"  # resized copy of the dataset (faster loading)
CACHE_MAX_SIDE = 1024  # longest image side after caching

# --- Model / training ---
BASE_CONFIG = "configs/dinov2/btxrd/instance/eomt_base_640.yaml"
RUN_NAME = "btxrd_instance_eomt_base_640"
INIT_FROM_COCO = True  # start from EoMT-B trained on COCO panoptic (recommended)
COCO_CKPT_REPO = "tue-mps/coco_panoptic_eomt_base_640_2x"
IMG_SIZE = 640  # must stay 640 when INIT_FROM_COCO=True
EPOCHS = 40
BATCH_SIZE_PER_GPU = 4  # 4 x 2 T4 = total batch 8
NUM_WORKERS = 4
LR = 1e-4
BALANCED_SAMPLING = True  # oversample the minority class (malignant)
COMPILE = False  # torch.compile; off by default for stability on Kaggle GPUs
SEED = 0
USE_WANDB = True  # only used if a WANDB_API_KEY secret is available

SMOKE_TEST = False  # True = 1 short epoch to check the pipeline end-to-end

CKPT_DIR = WORK_DIR / "checkpoints" / RUN_NAME
LOG_DIR = WORK_DIR / "logs"
RUN_CONFIG = WORK_DIR / f"{RUN_NAME}.yaml"
TEST_CONFIG = WORK_DIR / f"{RUN_NAME}_test.yaml"
BEST_WEIGHTS = WORK_DIR / f"{RUN_NAME}_best.bin"

## 2. Cài đặt môi trường

In [ ]:
# Versions pinned as in requirements.txt. torch/torchvision come from the Kaggle image.
PACKAGES = [
    "lightning==2.5.1.post0",
    "jsonargparse[signatures]==4.38",
    "timm==1.0.15",
    "transformers==4.56.1",
    "torchmetrics==1.7.1",
    "pycocotools==2.0.8",
    "gitignore_parser==0.1.12",
    "wandb==0.19.10",
]
if ON_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)

import torch

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
NUM_GPUS = torch.cuda.device_count()

## 3. Lấy mã nguồn dự án

Clone repo vào `/kaggle/working/eomt`. Nếu repo đã được thêm làm Kaggle Input (chạy offline), notebook sẽ sao chép từ đó.

In [ ]:
def find_input_repo():
    for pattern in ("*/models/eomt.py", "*/*/models/eomt.py"):
        for p in Path("/kaggle/input").glob(pattern):
            return p.parent.parent
    return None


if ON_KAGGLE and not (REPO_DIR / "main.py").exists():
    input_repo = find_input_repo()
    if input_repo is not None:
        shutil.copytree(input_repo, REPO_DIR)
        print("Copied repo from", input_repo)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# The repo's `datasets` package clashes with Hugging Face `datasets` if that was imported.
for name in [m for m in sys.modules if m == "datasets" or m.startswith("datasets.")]:
    del sys.modules[name]

(REPO_DIR / "configs/dinov2/btxrd/instance").mkdir(parents=True, exist_ok=True)
print("Repo:", REPO_DIR)
print(sorted(p.name for p in REPO_DIR.iterdir()))

## 4. Thêm các thành phần BTXRD vào dự án

Ba file dưới đây theo đúng chuẩn của repo, nên dùng được trực tiếp với `main.py`:
- `datasets/btxrd.py`: `BTXRDInstance` (LightningDataModule) đọc ảnh và polygon LabelMe, gán nhãn lành/ác theo `split.csv`, dùng chung `Transforms` của dự án. Có tùy chọn cân bằng lớp khi lấy mẫu.
- `training/mask_classification_btxrd.py`: `MaskClassificationBTXRD`, kế thừa `MaskClassificationInstance` (giữ mask AP), thêm Dice/IoU vùng u, các chỉ số phân loại lành/ác và `test_step`.
- `configs/dinov2/btxrd/instance/eomt_base_640.yaml`: cấu hình EoMT-B (DINOv2) 640×640.

Nếu repo đã có sẵn các file này, các ô bên dưới ghi đè bằng đúng nội dung đó.

In [ ]:
%%writefile datasets/btxrd.py
# ---------------------------------------------------------------
# BTXRD (Bone Tumor X-ray Radiograph Dataset) data module for EoMT.
#
# Bone tumors are cast as an instance segmentation problem with two classes:
#   0 = benign, 1 = malignant
# Every tumor polygon in a LabelMe annotation becomes one instance mask whose
# label is the benign/malignant type of the image. This lets EoMT segment the
# tumor and classify it as benign or malignant with the same query heads.
#
# Expected folder layout (as in the BTXRD release split into train/val/test):
#   <path>/split.csv                     image_id,split,type (benign|malignant)
#   <path>/{train,val,test}/images/*.jpeg
#   <path>/{train,val,test}/Annotations/*.json   (LabelMe format)
# ---------------------------------------------------------------

import csv
import json
import logging
from collections import Counter
from pathlib import Path
from typing import Optional, Union

import torch
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import tv_tensors

from datasets.lightning_data_module import LightningDataModule
from datasets.transforms import Transforms

CLASS_NAMES = ("benign", "malignant")
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

# Tumor subtypes used as LabelMe labels in BTXRD, mapped to benign/malignant.
# Only used as a fallback when the image type is not found in split.csv.
SUBTYPE_TO_CLASS = {
    "osteochondroma": 0,
    "multiple osteochondromas": 0,
    "simple bone cyst": 0,
    "giant cell tumor": 0,
    "osteofibroma": 0,
    "synovial osteochondroma": 0,
    "other bt": 0,
    "osteosarcoma": 1,
    "other mt": 1,
}

IMG_SUFFIXES = (".jpeg", ".jpg", ".png")


def load_image_types(path: Union[str, Path]) -> dict[str, int]:
    """Read the benign/malignant type of every image from split.csv."""
    csv_path = Path(path, "split.csv")
    types = {}
    if not csv_path.exists():
        return types

    with open(csv_path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            image_type = row.get("type", "").strip().lower()
            if image_type in CLASS_TO_ID:
                types[Path(row["image_id"]).stem] = CLASS_TO_ID[image_type]

    return types


def load_tumor_shapes(ann_path: Path) -> tuple[list[tuple[list, str]], int, int]:
    """Return the tumor polygons of a LabelMe file as (points, subtype) pairs.

    Polygons are the segmentation ground truth. Rectangles (bounding boxes) are
    only used for images that have no polygon at all.
    """
    with open(ann_path, encoding="utf-8") as f:
        ann = json.load(f)

    shapes = ann.get("shapes", [])
    polygons = [
        (s["points"], s["label"].strip().lower())
        for s in shapes
        if s.get("shape_type") == "polygon" and len(s["points"]) >= 3
    ]

    if not polygons:
        for s in shapes:
            if s.get("shape_type") == "rectangle" and len(s["points"]) == 2:
                (x0, y0), (x1, y1) = s["points"]
                polygons.append(
                    ([[x0, y0], [x1, y0], [x1, y1], [x0, y1]], s["label"].strip().lower())
                )

    return polygons, ann.get("imageHeight"), ann.get("imageWidth")


class BTXRDDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        root: Union[str, Path],
        split: str,
        image_types: dict[str, int],
        transforms: Optional[torch.nn.Module] = None,
        check_empty_targets: bool = True,
    ):
        self.root = Path(root)
        self.split = split
        self.transforms = transforms

        img_dir = self.root / split / "images"
        ann_dir = self.root / split / "Annotations"
        if not img_dir.is_dir():
            raise FileNotFoundError(f"Image folder not found: {img_dir}")

        self.samples: list[tuple[Path, Path, int]] = []
        skipped = Counter()

        for img_path in sorted(img_dir.iterdir()):
            if img_path.suffix.lower() not in IMG_SUFFIXES:
                continue

            ann_path = ann_dir / f"{img_path.stem}.json"
            if not ann_path.exists():
                skipped["no annotation"] += 1
                continue

            polygons, _, _ = load_tumor_shapes(ann_path)
            if check_empty_targets and not polygons:
                skipped["no tumor shape"] += 1
                continue

            cls_id = image_types.get(img_path.stem)
            if cls_id is None and polygons:
                cls_id = SUBTYPE_TO_CLASS.get(polygons[0][1])
            if cls_id is None:
                skipped["unknown type"] += 1
                continue

            self.samples.append((img_path, ann_path, cls_id))

        if skipped:
            logging.info(f"BTXRD/{split}: skipped {dict(skipped)}")

        if not self.samples:
            raise RuntimeError(f"No usable samples found in {self.root / split}")

    @property
    def class_ids(self) -> list[int]:
        return [cls_id for _, _, cls_id in self.samples]

    @staticmethod
    def rasterize(polygons, ann_h, ann_w, height, width) -> list[torch.Tensor]:
        # Annotations may refer to the original resolution; rescale if needed.
        sx = width / ann_w if ann_w else 1.0
        sy = height / ann_h if ann_h else 1.0

        masks = []
        for points, _ in polygons:
            canvas = Image.new("L", (width, height), 0)
            ImageDraw.Draw(canvas).polygon(
                [(x * sx, y * sy) for x, y in points], outline=1, fill=1
            )
            mask = torch.frombuffer(bytearray(canvas.tobytes()), dtype=torch.uint8)
            mask = mask.view(height, width).bool()
            if mask.any():
                masks.append(mask)

        return masks

    def __getitem__(self, index: int):
        img_path, ann_path, cls_id = self.samples[index]

        with Image.open(img_path) as pil_img:
            img = tv_tensors.Image(pil_img.convert("RGB"))

        height, width = img.shape[-2:]
        polygons, ann_h, ann_w = load_tumor_shapes(ann_path)
        masks = self.rasterize(polygons, ann_h, ann_w, height, width)

        if masks:
            masks = torch.stack(masks)
        elif self.transforms is not None:
            # Degenerate polygons: training transforms need at least one mask.
            return self[(index + 1) % len(self)]
        else:
            masks = torch.zeros((0, height, width), dtype=torch.bool)

        target = {
            "masks": tv_tensors.Mask(masks),
            "labels": torch.full((len(masks),), cls_id, dtype=torch.long),
            "is_crowd": torch.zeros(len(masks), dtype=torch.bool),
        }

        if self.transforms is not None:
            img, target = self.transforms(img, target)

        return img, target

    def __len__(self):
        return len(self.samples)


class BTXRDInstance(LightningDataModule):
    def __init__(
        self,
        path,
        num_workers: int = 4,
        batch_size: int = 8,
        img_size: tuple[int, int] = (640, 640),
        num_classes: int = 2,
        color_jitter_enabled: bool = True,
        scale_range: tuple[float, float] = (0.5, 1.5),
        check_empty_targets: bool = True,
        balanced_sampling: bool = False,
    ) -> None:
        super().__init__(
            path=path,
            batch_size=batch_size,
            num_workers=num_workers,
            num_classes=num_classes,
            img_size=img_size,
            check_empty_targets=check_empty_targets,
        )
        self.save_hyperparameters(ignore=["_class_path"])

        self.balanced_sampling = balanced_sampling
        self.class_names = CLASS_NAMES

        self.transforms = Transforms(
            img_size=img_size,
            color_jitter_enabled=color_jitter_enabled,
            scale_range=scale_range,
        )

    def setup(self, stage: Union[str, None] = None) -> LightningDataModule:
        image_types = load_image_types(self.path)
        dataset_kwargs = {
            "root": self.path,
            "image_types": image_types,
            "check_empty_targets": self.check_empty_targets,
        }

        self.train_dataset = BTXRDDataset(
            split="train", transforms=self.transforms, **dataset_kwargs
        )
        self.val_dataset = BTXRDDataset(split="val", **dataset_kwargs)

        self.test_dataset = None
        if Path(self.path, "test", "images").is_dir():
            self.test_dataset = BTXRDDataset(split="test", **dataset_kwargs)

        return self

    def train_dataloader(self):
        sampler, shuffle = None, True
        if self.balanced_sampling:
            class_ids = self.train_dataset.class_ids
            counts = Counter(class_ids)
            weights = [1.0 / counts[c] for c in class_ids]
            sampler = WeightedRandomSampler(
                weights, num_samples=len(weights), replacement=True
            )
            shuffle = False

        return DataLoader(
            self.train_dataset,
            shuffle=shuffle,
            sampler=sampler,
            drop_last=True,
            collate_fn=self.train_collate,
            **self.dataloader_kwargs,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            collate_fn=self.eval_collate,
            **self.dataloader_kwargs,
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            collate_fn=self.eval_collate,
            **self.dataloader_kwargs,
        )

In [ ]:
%%writefile training/mask_classification_btxrd.py
# ---------------------------------------------------------------
# EoMT for bone tumor segmentation + benign/malignant classification (BTXRD).
#
# Extends MaskClassificationInstance (mask AP per layer) with metrics that
# matter clinically, computed on the final layer:
#   - tumor segmentation: Dice and IoU of the predicted tumor region
#   - image-level classification (malignant = positive): accuracy, balanced
#     accuracy, AUROC, F1, sensitivity, specificity
# It also adds test_step so `python main.py test` works on the test split.
# ---------------------------------------------------------------


from typing import List, Optional
import torch
import torch.nn as nn
import torch.nn.functional as F
from lightning.fabric.utilities import rank_zero_info
from torchmetrics import MeanMetric, MetricCollection
from torchmetrics.classification import (
    BinaryAccuracy,
    BinaryAUROC,
    BinaryF1Score,
    BinaryRecall,
    BinarySpecificity,
)

from training.mask_classification_instance import MaskClassificationInstance
from training.lightning_module import bold_green, reset


class MaskClassificationBTXRD(MaskClassificationInstance):
    def __init__(
        self,
        network: nn.Module,
        img_size: tuple[int, int],
        num_classes: int,
        attn_mask_annealing_enabled: bool,
        attn_mask_annealing_start_steps: Optional[list[int]] = None,
        attn_mask_annealing_end_steps: Optional[list[int]] = None,
        lr: float = 1e-4,
        llrd: float = 0.8,
        llrd_l2_enabled: bool = True,
        lr_mult: float = 1.0,
        weight_decay: float = 0.05,
        num_points: int = 12544,
        oversample_ratio: float = 3.0,
        importance_sample_ratio: float = 0.75,
        poly_power: float = 0.9,
        warmup_steps: List[int] = [500, 1000],
        no_object_coefficient: float = 0.1,
        mask_coefficient: float = 5.0,
        dice_coefficient: float = 5.0,
        class_coefficient: float = 2.0,
        mask_thresh: float = 0.8,
        overlap_thresh: float = 0.8,
        eval_top_k_instances: int = 100,
        tumor_score_thresh: float = 0.5,
        ckpt_path: Optional[str] = None,
        delta_weights: bool = False,
        load_ckpt_class_head: bool = True,
    ):
        super().__init__(
            network=network,
            img_size=img_size,
            num_classes=num_classes,
            attn_mask_annealing_enabled=attn_mask_annealing_enabled,
            attn_mask_annealing_start_steps=attn_mask_annealing_start_steps,
            attn_mask_annealing_end_steps=attn_mask_annealing_end_steps,
            lr=lr,
            llrd=llrd,
            llrd_l2_enabled=llrd_l2_enabled,
            lr_mult=lr_mult,
            weight_decay=weight_decay,
            num_points=num_points,
            oversample_ratio=oversample_ratio,
            importance_sample_ratio=importance_sample_ratio,
            poly_power=poly_power,
            warmup_steps=warmup_steps,
            no_object_coefficient=no_object_coefficient,
            mask_coefficient=mask_coefficient,
            dice_coefficient=dice_coefficient,
            class_coefficient=class_coefficient,
            mask_thresh=mask_thresh,
            overlap_thresh=overlap_thresh,
            eval_top_k_instances=eval_top_k_instances,
            ckpt_path=ckpt_path,
            delta_weights=delta_weights,
            load_ckpt_class_head=load_ckpt_class_head,
        )

        if num_classes != 2:
            raise ValueError("BTXRD expects num_classes=2 (benign, malignant)")

        self.tumor_score_thresh = tumor_score_thresh

        self.cls_metrics = MetricCollection(
            {
                "acc": BinaryAccuracy(),
                "auroc": BinaryAUROC(),
                "f1": BinaryF1Score(),
                "sensitivity": BinaryRecall(),
                "specificity": BinarySpecificity(),
            }
        )
        self.dice_metric = MeanMetric()
        self.iou_metric = MeanMetric()

    @staticmethod
    def predict_btxrd(mask_logits, class_logits, tumor_score_thresh=0.5):
        """Turn the query outputs of one image into BTXRD predictions.

        Args:
            mask_logits: (Q, H, W) mask logits at image resolution.
            class_logits: (Q, num_classes + 1) class logits.

        Returns:
            tumor_mask: (H, W) bool, union of the masks of all confident tumor
                queries (the most confident query is always kept).
            malignant_prob: scalar, P(malignant | tumor) of the top query.
            top_query: index of the most confident tumor query.
        """
        probs = class_logits.float().softmax(dim=-1)
        tumor_scores = probs[:, :-1].sum(dim=-1)
        top_query = tumor_scores.argmax()

        malignant_prob = probs[top_query, 1] / tumor_scores[top_query].clamp_min(1e-6)

        keep = tumor_scores > tumor_score_thresh
        keep[top_query] = True
        tumor_mask = (mask_logits[keep] > 0).any(dim=0)

        return tumor_mask, malignant_prob, top_query

    @torch.compiler.disable
    def update_metrics_btxrd(self, mask_logits, class_logits, targets):
        for j in range(len(mask_logits)):
            tumor_mask, malignant_prob, _ = self.predict_btxrd(
                mask_logits[j], class_logits[j], self.tumor_score_thresh
            )

            gt_mask = targets[j]["masks"].bool().any(dim=0)
            gt_malignant = (targets[j]["labels"] == 1).any().long()

            inter = (tumor_mask & gt_mask).sum().float()
            pred_area = tumor_mask.sum().float()
            gt_area = gt_mask.sum().float()
            union = (tumor_mask | gt_mask).sum().float()

            self.dice_metric.update(2 * inter / (pred_area + gt_area).clamp_min(1))
            self.iou_metric.update(inter / union.clamp_min(1))
            self.cls_metrics.update(malignant_prob[None], gt_malignant[None])

    def eval_step(
        self,
        batch,
        batch_idx=None,
        log_prefix=None,
    ):
        imgs, targets = batch

        img_sizes = [img.shape[-2:] for img in imgs]
        transformed_imgs = self.resize_and_pad_imgs_instance_panoptic(imgs)
        mask_logits_per_layer, class_logits_per_layer = self(transformed_imgs)
        last_layer = len(mask_logits_per_layer) - 1

        for i, (mask_logits, class_logits) in enumerate(
            list(zip(mask_logits_per_layer, class_logits_per_layer))
        ):
            mask_logits = F.interpolate(mask_logits, self.img_size, mode="bilinear")
            mask_logits = self.revert_resize_and_pad_logits_instance_panoptic(
                mask_logits, img_sizes
            )

            if i == last_layer:
                self.update_metrics_btxrd(mask_logits, class_logits, targets)

            # Same mask-AP evaluation as MaskClassificationInstance.eval_step
            preds, targets_ = [], []
            for j in range(len(mask_logits)):
                scores = class_logits[j].softmax(dim=-1)[:, :-1]
                labels = (
                    torch.arange(scores.shape[-1], device=self.device)
                    .unsqueeze(0)
                    .repeat(scores.shape[0], 1)
                    .flatten(0, 1)
                )

                topk_scores, topk_indices = scores.flatten(0, 1).topk(
                    min(self.eval_top_k_instances, scores.numel()), sorted=False
                )
                labels = labels[topk_indices]

                topk_indices = topk_indices // scores.shape[-1]
                layer_mask_logits = mask_logits[j][topk_indices]

                masks = layer_mask_logits > 0
                mask_scores = (
                    layer_mask_logits.sigmoid().flatten(1) * masks.flatten(1)
                ).sum(1) / (masks.flatten(1).sum(1) + 1e-6)
                scores = topk_scores * mask_scores

                preds.append(dict(masks=masks, labels=labels, scores=scores))
                targets_.append(
                    dict(
                        masks=targets[j]["masks"],
                        labels=targets[j]["labels"],
                        iscrowd=targets[j]["is_crowd"],
                    )
                )

            self.update_metrics_instance(preds, targets_, i)

    def test_step(self, batch, batch_idx=0):
        return self.eval_step(batch, batch_idx, "test")

    def _on_eval_epoch_end_btxrd(self, log_prefix):
        cls_results = self.cls_metrics.compute()
        self.cls_metrics.reset()
        for name, value in cls_results.items():
            self.log(f"metrics/{log_prefix}_cls_{name}", value)

        balanced_acc = (cls_results["sensitivity"] + cls_results["specificity"]) / 2
        self.log(f"metrics/{log_prefix}_cls_balanced_acc", balanced_acc)

        dice = self.dice_metric.compute()
        iou = self.iou_metric.compute()
        self.dice_metric.reset()
        self.iou_metric.reset()
        self.log(f"metrics/{log_prefix}_dice", dice)
        self.log(f"metrics/{log_prefix}_iou", iou)

        # Single number for checkpoint selection: segmentation + classification
        self.log(f"metrics/{log_prefix}_score", (dice + cls_results["auroc"]) / 2)

    def _on_eval_end_btxrd(self, log_prefix):
        if self.trainer.sanity_checking:
            return

        m = self.trainer.callback_metrics
        rank_zero_info(
            f"{bold_green}Dice: {m[f'metrics/{log_prefix}_dice'] * 100:.1f} | "
            f"IoU: {m[f'metrics/{log_prefix}_iou'] * 100:.1f} | "
            f"Acc: {m[f'metrics/{log_prefix}_cls_acc'] * 100:.1f} | "
            f"Bal. Acc: {m[f'metrics/{log_prefix}_cls_balanced_acc'] * 100:.1f} | "
            f"AUROC: {m[f'metrics/{log_prefix}_cls_auroc'] * 100:.1f} | "
            f"Sens: {m[f'metrics/{log_prefix}_cls_sensitivity'] * 100:.1f} | "
            f"Spec: {m[f'metrics/{log_prefix}_cls_specificity'] * 100:.1f}{reset}"
        )

    def on_validation_epoch_end(self):
        self._on_eval_epoch_end_instance("val")
        self._on_eval_epoch_end_btxrd("val")

    def on_validation_end(self):
        self._on_eval_end_instance("val")
        self._on_eval_end_btxrd("val")

    def on_test_epoch_end(self):
        self._on_eval_epoch_end_instance("test")
        self._on_eval_epoch_end_btxrd("test")

    def on_test_end(self):
        self._on_eval_end_instance("test")
        self._on_eval_end_btxrd("test")

In [ ]:
%%writefile configs/dinov2/btxrd/instance/eomt_base_640.yaml
# EoMT-B (DINOv2) on BTXRD: bone tumor segmentation + benign/malignant classification.
# Annealing/warmup steps assume ~1,300 training images, batch size 8, 40 epochs
# (~6,500 steps). Rescale them when the dataset size, batch size or epochs change
# (the Kaggle notebook btxrd_eomt_kaggle.ipynb does this automatically).
trainer:
  max_epochs: 40
  check_val_every_n_epoch: 1
  logger:
    class_path: lightning.pytorch.loggers.wandb.WandbLogger
    init_args:
      resume: allow
      project: "eomt"
      name: "btxrd_instance_eomt_base_640"
  callbacks:
    - class_path: lightning.pytorch.callbacks.ModelCheckpoint
      init_args:
        monitor: metrics/val_score
        mode: max
        save_last: true
        save_top_k: 1
        filename: "best-epoch{epoch:02d}"
        auto_insert_metric_name: false
model:
  class_path: training.mask_classification_btxrd.MaskClassificationBTXRD
  init_args:
    attn_mask_annealing_enabled: True
    attn_mask_annealing_start_steps: [1080, 2700, 4320]
    attn_mask_annealing_end_steps: [2160, 3780, 5400]
    warmup_steps: [200, 400]
    network:
      class_path: models.eomt.EoMT
      init_args:
        num_q: 200
        num_blocks: 3
        encoder:
          class_path: models.vit.ViT
          init_args:
            backbone_name: vit_base_patch14_reg4_dinov2
data:
  class_path: datasets.btxrd.BTXRDInstance
  init_args:
    batch_size: 8
    img_size: [640, 640]
    balanced_sampling: true

## 5. Dữ liệu BTXRD: khám phá

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from PIL import Image

from datasets.btxrd import (
    CLASS_NAMES,
    IMG_SUFFIXES,
    load_image_types,
    load_tumor_shapes,
)


def find_data_root():
    if DATA_ROOT is not None:
        return Path(DATA_ROOT)
    if not ON_KAGGLE:
        return REPO_DIR / "BTXRD"
    for pattern in ("split.csv", "*/split.csv", "*/*/split.csv", "*/*/*/split.csv"):
        hits = sorted(Path("/kaggle/input").glob(pattern))
        if hits:
            return hits[0].parent
    raise FileNotFoundError("split.csv not found under /kaggle/input; set DATA_ROOT")


SRC_ROOT = find_data_root()
print("Dataset:", SRC_ROOT)

split_df = pd.read_csv(SRC_ROOT / "split.csv")
display(pd.crosstab(split_df["split"], split_df["type"], margins=True))


def list_source_items(root):
    """(split, image path, annotation path) for both supported layouts."""
    items = []
    split_dirs = [s for s in ("train", "val", "test") if (root / s / "images").is_dir()]
    if split_dirs:
        for s in split_dirs:
            for img in sorted((root / s / "images").iterdir()):
                if img.suffix.lower() in IMG_SUFFIXES:
                    items.append((s, img, root / s / "Annotations" / f"{img.stem}.json"))
    else:  # flat layout: images/ + Annotations/ + split column in split.csv
        split_of = {Path(i).stem: s for i, s in zip(split_df["image_id"], split_df["split"])}
        for img in sorted((root / "images").iterdir()):
            s = split_of.get(img.stem)
            if s and img.suffix.lower() in IMG_SUFFIXES:
                items.append((s, img, root / "Annotations" / f"{img.stem}.json"))
    return items


source_items = list_source_items(SRC_ROOT)
image_types = load_image_types(SRC_ROOT)

rows = []
for split, img_path, ann_path in source_items:
    polygons, h, w = load_tumor_shapes(ann_path) if ann_path.exists() else ([], None, None)
    rows.append(
        dict(
            split=split,
            image=img_path.stem,
            type=CLASS_NAMES[image_types[img_path.stem]] if img_path.stem in image_types else None,
            subtype=polygons[0][1] if polygons else None,
            n_tumors=len(polygons),
            height=h,
            width=w,
        )
    )
meta_df = pd.DataFrame(rows)
print(f"{len(meta_df)} images | missing annotation: {(meta_df.n_tumors == 0).sum()}")
display(meta_df.groupby(["type", "subtype"]).size().rename("images").to_frame())
display(meta_df[["n_tumors", "height", "width"]].describe().round(1))

In [ ]:
def show_samples(df, n_per_class=3):
    picks = pd.concat(
        [g.sample(min(n_per_class, len(g)), random_state=SEED) for _, g in df.dropna(subset=["type"]).groupby("type")]
    )
    fig, axes = plt.subplots(1, len(picks), figsize=(4 * len(picks), 5))
    for ax, (_, r) in zip(np.atleast_1d(axes), picks.iterrows()):
        split, img_path, ann_path = next(it for it in source_items if it[1].stem == r.image)
        with Image.open(img_path) as im:
            ax.imshow(im.convert("L"), cmap="gray")
        color = "tab:red" if r.type == "malignant" else "tab:green"
        for points, _ in load_tumor_shapes(ann_path)[0]:
            ax.add_patch(MplPolygon(points, closed=True, fill=False, ec=color, lw=2))
        ax.set_title(f"{r.image}\n{r.type} / {r.subtype}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


show_samples(meta_df[meta_df.n_tumors > 0])

## 6. Tiền xử lý: cache ảnh đã thu nhỏ

Ảnh X-quang gốc khoảng 2000–3000 px, giải mã JPEG mỗi bước huấn luyện sẽ rất chậm. Ta thu nhỏ một lần sao cho cạnh dài ≤ `CACHE_MAX_SIDE` (vẫn đủ lớn cho scale jitter ở 640) và đổi tọa độ polygon theo cùng tỉ lệ. Cấu trúc thư mục cache giống hệt `train/val/test` của BTXRD.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

DONE_MARKER = CACHE_DIR / f".done_{CACHE_MAX_SIDE}"


def cache_one(item):
    split, img_path, ann_path = item
    out_img = CACHE_DIR / split / "images" / f"{img_path.stem}.jpeg"
    out_ann = CACHE_DIR / split / "Annotations" / f"{img_path.stem}.json"

    with Image.open(img_path) as im:
        orig_w, orig_h = im.size
        im.draft("RGB", (CACHE_MAX_SIDE, CACHE_MAX_SIDE))  # fast JPEG downscale
        im = im.convert("L" if im.mode in ("L", "I;16", "I") else "RGB")
        im.thumbnail((CACHE_MAX_SIDE, CACHE_MAX_SIDE), Image.LANCZOS)
        new_w, new_h = im.size
        im.save(out_img, quality=95)

    if ann_path.exists():
        with open(ann_path, encoding="utf-8") as f:
            ann = json.load(f)
        # Polygons are relative to the size stored in the json (fallback: real size)
        sx = new_w / (ann.get("imageWidth") or orig_w)
        sy = new_h / (ann.get("imageHeight") or orig_h)
        for shape in ann.get("shapes", []):
            shape["points"] = [[x * sx, y * sy] for x, y in shape["points"]]
        ann.update(imageWidth=new_w, imageHeight=new_h, imageData=None, imagePath=out_img.name)
        with open(out_ann, "w", encoding="utf-8") as f:
            json.dump(ann, f)


if not DONE_MARKER.exists():
    for split in {s for s, _, _ in source_items}:
        (CACHE_DIR / split / "images").mkdir(parents=True, exist_ok=True)
        (CACHE_DIR / split / "Annotations").mkdir(parents=True, exist_ok=True)
    with ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as pool:
        list(pool.map(cache_one, source_items))
    for extra in ("split.csv", "dataset.xlsx"):
        if (SRC_ROOT / extra).exists():
            shutil.copy(SRC_ROOT / extra, CACHE_DIR / extra)
    DONE_MARKER.touch()

for split in ("train", "val", "test"):
    if (CACHE_DIR / split).exists():
        print(split, len(list((CACHE_DIR / split / "images").iterdir())), "images")

## 7. Kiểm tra DataModule

Khởi tạo `BTXRDInstance` như `main.py` sẽ làm, rồi xem vài mẫu train sau augmentation (scale jitter, flip, crop, color jitter) và một mẫu val.

In [ ]:
from collections import Counter
from datasets.btxrd import BTXRDInstance

data = BTXRDInstance(
    path=str(CACHE_DIR),
    batch_size=BATCH_SIZE_PER_GPU,
    num_workers=0,
    img_size=(IMG_SIZE, IMG_SIZE),
).setup()

for name in ("train", "val", "test"):
    ds = getattr(data, f"{name}_dataset")
    if ds is not None:
        counts = Counter(CLASS_NAMES[c] for c in ds.class_ids)
        print(f"{name:5s}: {len(ds):5d} images  {dict(counts)}")

CLASS_COLORS = {0: np.array([0.1, 0.8, 0.2]), 1: np.array([0.9, 0.1, 0.1])}


def overlay(ax, img, masks, labels, title):
    ax.imshow(img.permute(1, 2, 0).numpy())
    for m, l in zip(masks, labels):
        rgba = np.zeros((*m.shape, 4))
        rgba[m.numpy().astype(bool)] = [*CLASS_COLORS[int(l)], 0.4]
        ax.imshow(rgba)
    ax.set_title(title, fontsize=9)
    ax.axis("off")


fig, axes = plt.subplots(1, 5, figsize=(20, 4.5))
for k, ax in enumerate(axes[:4]):
    img, tgt = data.train_dataset[k * 7 % len(data.train_dataset)]
    overlay(ax, img, tgt["masks"], tgt["labels"], f"train (aug) {tuple(img.shape[-2:])} {CLASS_NAMES[int(tgt['labels'][0])]}")
img, tgt = data.val_dataset[0]
overlay(axes[4], img, tgt["masks"], tgt["labels"], f"val {tuple(img.shape[-2:])} {CLASS_NAMES[int(tgt['labels'][0])]}")
plt.tight_layout()
plt.show()

## 8. Tạo file cấu hình cho lần chạy

Bắt đầu từ `configs/dinov2/btxrd/instance/eomt_base_640.yaml`, cập nhật theo kích thước dữ liệu và phần cứng:
- `attn_mask_annealing_*_steps` và `warmup_steps` co giãn theo tổng số bước, cùng tỉ lệ với config COCO/ADE20K gốc (annealing kết thúc ở khoảng 5/6 quá trình huấn luyện).
- Fine-tune từ checkpoint EoMT-B COCO panoptic (`--model.ckpt_path`, `--model.load_ckpt_class_head False`) như hướng dẫn trong README.
- Nhiều GPU thì dùng DDP; logger là CSV, thêm W&B nếu có `WANDB_API_KEY`.

In [ ]:
import yaml

with open(REPO_DIR / BASE_CONFIG) as f:
    cfg = yaml.safe_load(f)

devices = max(1, NUM_GPUS)
steps_per_epoch = len(data.train_dataset) // (BATCH_SIZE_PER_GPU * devices)
epochs = 1 if SMOKE_TEST else EPOCHS
total_steps = max(1, steps_per_epoch * epochs)

# Mask-annealing schedule, as in the COCO/ADE20K configs: one start per masked block,
# spread over [1/6, 4/6] of training, each lasting 1/6 of training.
num_blocks = cfg["model"]["init_args"]["network"]["init_args"].get("num_blocks", 4)
unit = total_steps / 6
starts = [round(unit + k * 3 * unit / max(1, num_blocks - 1)) for k in range(num_blocks)]
ends = [round(s + unit) for s in starts]
warmup = [min(500, max(1, round(0.05 * total_steps))), min(1000, max(1, round(0.10 * total_steps)))]

model_args = cfg["model"]["init_args"]
model_args.update(
    lr=LR,
    warmup_steps=warmup,
    attn_mask_annealing_start_steps=starts,
    attn_mask_annealing_end_steps=ends,
)

if INIT_FROM_COCO:
    assert IMG_SIZE == 640, "The COCO EoMT-B checkpoint is trained at 640x640"
    from huggingface_hub import hf_hub_download

    model_args["ckpt_path"] = hf_hub_download(COCO_CKPT_REPO, "pytorch_model.bin")
    model_args["load_ckpt_class_head"] = False  # 133 COCO classes -> 2 BTXRD classes

cfg["data"]["init_args"].update(
    path=str(CACHE_DIR),
    batch_size=BATCH_SIZE_PER_GPU,
    num_workers=NUM_WORKERS,
    img_size=[IMG_SIZE, IMG_SIZE],
    balanced_sampling=BALANCED_SAMPLING,
)

# Loggers: CSV always (read back below); W&B first if a key is available,
# so main.py's `logger.experiment.log_code` keeps working as in the original project.
loggers = [
    {
        "class_path": "lightning.pytorch.loggers.CSVLogger",
        "init_args": {"save_dir": str(LOG_DIR), "name": RUN_NAME},
    }
]
wandb_key = os.environ.get("WANDB_API_KEY")
if ON_KAGGLE and USE_WANDB and not wandb_key:
    try:
        from kaggle_secrets import UserSecretsClient

        wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception:
        wandb_key = None
if USE_WANDB and wandb_key:
    os.environ["WANDB_API_KEY"] = wandb_key
    wandb_logger = cfg["trainer"]["logger"]
    wandb_logger["init_args"].update(name=RUN_NAME, save_dir=str(WORK_DIR))
    loggers.insert(0, wandb_logger)
print("Loggers:", [l["class_path"].rsplit(".", 1)[-1] for l in loggers])

trainer = cfg["trainer"]
trainer.update(
    max_epochs=epochs,
    devices=devices,
    accelerator="gpu" if NUM_GPUS else "cpu",
    precision="16-mixed" if NUM_GPUS else "32-true",
    logger=loggers,
    log_every_n_steps=min(50, max(1, steps_per_epoch)),
)
if devices > 1:
    trainer["strategy"] = "ddp"
for cb in trainer["callbacks"]:
    if cb["class_path"].endswith("ModelCheckpoint"):
        cb["init_args"]["dirpath"] = str(CKPT_DIR)
if SMOKE_TEST:
    trainer.update(limit_train_batches=2, limit_val_batches=2, num_sanity_val_steps=0)
cfg["seed_everything"] = SEED

with open(RUN_CONFIG, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"train images={len(data.train_dataset)}, devices={devices}, total batch={BATCH_SIZE_PER_GPU * devices}")
print(f"steps/epoch={steps_per_epoch}, total steps={total_steps}, warmup={warmup}")
print(f"annealing starts={starts} ends={ends}")
print(RUN_CONFIG.read_text())

## 9. Huấn luyện

Chạy đúng lệnh của dự án (`main.py fit`) trong một tiến trình con, để Lightning dùng được DDP trên 2 GPU. Log được in trực tiếp ra bên dưới; checkpoint tốt nhất (theo `metrics/val_score`) và `last.ckpt` được lưu ở `CKPT_DIR`.

> Kaggle giới hạn 12 giờ mỗi phiên. Với khoảng 1.300 ảnh train, 40 epoch EoMT-B 640 trên 2×T4 mất cỡ vài giờ. Có thể giảm `EPOCHS` nếu cần.

In [ ]:
def run(cmd):
    """Run a command and stream its output into the notebook."""
    print("$", " ".join(map(str, cmd)), flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        list(map(str, cmd)), cwd=REPO_DIR, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}")


fit_cmd = [sys.executable, "main.py", "fit", "-c", RUN_CONFIG]
if not COMPILE:
    fit_cmd.append("--compile_disabled")
run(fit_cmd)

## 10. Đường cong huấn luyện và xuất trọng số tốt nhất

Checkpoint Lightning (`.ckpt`) được chuyển thành state dict thuần (`.bin`), cùng định dạng `pytorch_model.bin` mà README dùng cho `--model.ckpt_path`.

In [ ]:
best_ckpts = sorted(CKPT_DIR.glob("best-*.ckpt"), key=os.path.getmtime)
best_ckpt = best_ckpts[-1] if best_ckpts else CKPT_DIR / "last.ckpt"
print("Best checkpoint:", best_ckpt)

ckpt = torch.load(best_ckpt, map_location="cpu", weights_only=False)
state_dict = {k.replace("._orig_mod", ""): v for k, v in ckpt["state_dict"].items()}
torch.save(state_dict, BEST_WEIGHTS)
print(f"Saved {len(state_dict)} tensors -> {BEST_WEIGHTS} (epoch {ckpt.get('epoch')})")
del ckpt

metrics_csv = sorted(LOG_DIR.glob(f"{RUN_NAME}/version_*/metrics.csv"), key=os.path.getmtime)[-1]
hist = pd.read_csv(metrics_csv)
val = hist.dropna(subset=["metrics/val_dice"]).groupby("epoch").last()
loss_col = "losses/train_loss_total"  # logged every `log_every_n_steps` steps
train = hist.dropna(subset=[loss_col]).groupby("epoch")[loss_col].mean() if loss_col in hist else pd.Series(dtype=float)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
axes[0].plot(train.index, train.values, marker="o")
axes[0].set_title("Train loss (total)")
for col in ["metrics/val_dice", "metrics/val_iou", "metrics/val_ap_all"]:
    if col in val:
        axes[1].plot(val.index, val[col], marker="o", label=col.split("/")[-1])
axes[1].set_title("Segmentation (val)")
for col in ["metrics/val_cls_auroc", "metrics/val_cls_balanced_acc", "metrics/val_cls_sensitivity", "metrics/val_cls_specificity"]:
    if col in val:
        axes[2].plot(val.index, val[col], marker="o", label=col.split("/")[-1].replace("val_cls_", ""))
axes[2].set_title("Benign / malignant classification (val)")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
axes[1].legend()
axes[2].legend()
plt.tight_layout()
plt.show()

display(val[[c for c in val.columns if c.startswith("metrics/val_") and "block" not in c]].round(4).tail(5))

## 11. Đánh giá trên tập test bằng CLI của dự án

Giống phần *Evaluating* trong README: tắt masked attention khi suy luận (`masked_attn_enabled: False`) và nạp trọng số tốt nhất qua `model.ckpt_path`. Chạy trên 1 GPU để không bị lặp mẫu do DDP.

In [ ]:
test_cfg = yaml.safe_load(open(RUN_CONFIG))
test_cfg["model"]["init_args"].update(ckpt_path=str(BEST_WEIGHTS), load_ckpt_class_head=True)
test_cfg["model"]["init_args"]["network"]["init_args"]["masked_attn_enabled"] = False
test_cfg["trainer"].update(
    devices=1,
    logger=[{"class_path": "lightning.pytorch.loggers.CSVLogger",
             "init_args": {"save_dir": str(LOG_DIR), "name": f"{RUN_NAME}_test"}}],
)
test_cfg["trainer"].pop("strategy", None)
for key in ("limit_train_batches", "limit_val_batches"):
    test_cfg["trainer"].pop(key, None)
with open(TEST_CONFIG, "w") as f:
    yaml.safe_dump(test_cfg, f, sort_keys=False)

has_test = (CACHE_DIR / "test" / "images").is_dir()
run([sys.executable, "main.py", "test" if has_test else "validate", "-c", TEST_CONFIG])

## 12. Phân tích chi tiết và trực quan hóa trên tập test

Nạp mô hình trong notebook theo cách của `inference.ipynb`, dự đoán từng ảnh bằng chính hàm `MaskClassificationBTXRD.predict_btxrd` dùng khi tính metric. Sau đó vẽ confusion matrix, ROC, phân bố Dice và các ca dự đoán.

In [ ]:
import importlib
import warnings
import torch.nn.functional as F
from torch.amp import autocast

warnings.filterwarnings("ignore", message=r".*is an instance of `nn\.Module` and is already saved.*")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def build_model(config, weights):
    net_cfg = config["model"]["init_args"]["network"]
    enc_cfg = net_cfg["init_args"]["encoder"]
    img_size = tuple(config["data"]["init_args"]["img_size"])
    num_classes = config["data"]["init_args"].get("num_classes", 2)

    def load_cls(path):
        module, name = path.rsplit(".", 1)
        return getattr(importlib.import_module(module), name)

    encoder = load_cls(enc_cfg["class_path"])(
        img_size=img_size, ckpt_path=str(weights), **enc_cfg.get("init_args", {})
    )
    net_kwargs = {k: v for k, v in net_cfg["init_args"].items() if k not in ("encoder", "masked_attn_enabled")}
    network = load_cls(net_cfg["class_path"])(
        encoder=encoder, num_classes=num_classes, masked_attn_enabled=False, **net_kwargs
    )
    model_kwargs = {
        k: v for k, v in config["model"]["init_args"].items()
        if k not in ("network", "ckpt_path", "load_ckpt_class_head")
    }
    model = load_cls(config["model"]["class_path"])(
        network=network, img_size=img_size, num_classes=num_classes,
        ckpt_path=str(weights), load_ckpt_class_head=True, **model_kwargs,
    )
    return model.eval().to(device)


model = build_model(test_cfg, BEST_WEIGHTS)


@torch.no_grad()
def predict(img):
    """img: uint8 tensor (3, H, W) -> (tumor mask (H, W), P(malignant))."""
    with autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
        imgs = [img.to(device)]
        sizes = [img.shape[-2:]]
        x = model.resize_and_pad_imgs_instance_panoptic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(x)
        mask_logits = F.interpolate(mask_logits_per_layer[-1], model.img_size, mode="bilinear")
        mask_logits = model.revert_resize_and_pad_logits_instance_panoptic(mask_logits, sizes)[0]
        mask, p_malignant, _ = model.predict_btxrd(
            mask_logits.float(), class_logits_per_layer[-1][0], model.tumor_score_thresh
        )
    return mask.cpu(), float(p_malignant)


eval_ds = data.test_dataset if data.test_dataset is not None else data.val_dataset
records, cache = [], {}
for idx in range(len(eval_ds)):
    img, tgt = eval_ds[idx]
    mask, p_mal = predict(img)
    gt = tgt["masks"].bool().any(0)
    inter = (mask & gt).sum().item()
    records.append(
        dict(
            image=eval_ds.samples[idx][0].stem,
            label=int(tgt["labels"][0]),
            p_malignant=p_mal,
            pred=int(p_mal >= 0.5),
            dice=2 * inter / max(1, mask.sum().item() + gt.sum().item()),
            iou=inter / max(1, (mask | gt).sum().item()),
        )
    )
    cache[idx] = mask

pred_df = pd.DataFrame(records)
pred_df.to_csv(WORK_DIR / f"{RUN_NAME}_test_predictions.csv", index=False)
print(f"{len(pred_df)} images | mean Dice {pred_df.dice.mean():.4f} | mean IoU {pred_df.iou.mean():.4f}")
display(pred_df.groupby(pred_df.label.map(dict(enumerate(CLASS_NAMES))))[["dice", "iou"]].mean().round(4))

In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    classification_report,
    roc_auc_score,
)

print(classification_report(pred_df.label, pred_df.pred, target_names=CLASS_NAMES, digits=4, zero_division=0))
if pred_df.label.nunique() == 2:
    print(f"AUROC: {roc_auc_score(pred_df.label, pred_df.p_malignant):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ConfusionMatrixDisplay.from_predictions(
    pred_df.label, pred_df.pred, labels=[0, 1], display_labels=CLASS_NAMES, cmap="Blues", ax=axes[0]
)
axes[0].set_title("Confusion matrix")
if pred_df.label.nunique() == 2:
    RocCurveDisplay.from_predictions(pred_df.label, pred_df.p_malignant, name="EoMT", ax=axes[1])
    axes[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[1].set_title("ROC (malignant = positive)")
for c, name in enumerate(CLASS_NAMES):
    axes[2].hist(pred_df.dice[pred_df.label == c], bins=20, range=(0, 1), alpha=0.6, label=name)
axes[2].set_title("Tumor Dice per image")
axes[2].set_xlabel("Dice")
axes[2].legend()
plt.tight_layout()
plt.show()

In [ ]:
def show_predictions(indices, title):
    fig, axes = plt.subplots(len(indices), 3, figsize=(13, 4.3 * len(indices)), squeeze=False)
    for row, idx in zip(axes, indices):
        img, tgt = eval_ds[idx]
        r = pred_df.iloc[idx]
        base = img.permute(1, 2, 0).numpy()
        gt = tgt["masks"].bool().any(0).numpy()
        pred = cache[idx].numpy()

        row[0].imshow(base)
        row[0].set_title(r.image, fontsize=9)
        for ax, m, lbl, txt in (
            (row[1], gt, r.label, f"Ground truth: {CLASS_NAMES[r.label]}"),
            (row[2], pred, r.pred, f"Prediction: {CLASS_NAMES[r.pred]} (P_mal={r.p_malignant:.2f}) Dice={r.dice:.2f}"),
        ):
            rgba = np.zeros((*m.shape, 4))
            rgba[m] = [*CLASS_COLORS[int(lbl)], 0.45]
            ax.imshow(base)
            ax.imshow(rgba)
            ax.contour(gt, levels=[0.5], colors="yellow", linewidths=1)
            ax.set_title(txt, fontsize=9)
        for ax in row:
            ax.axis("off")
    fig.suptitle(title + "  (green = benign, red = malignant, yellow contour = ground truth)", y=1.0)
    plt.tight_layout()
    plt.show()


rng = np.random.default_rng(SEED)
show_predictions(rng.choice(len(pred_df), size=min(4, len(pred_df)), replace=False), "Random test images")

wrong = pred_df.index[pred_df.label != pred_df.pred].tolist()
if wrong:
    show_predictions(wrong[:4], "Misclassified (benign <-> malignant)")
show_predictions(pred_df.nsmallest(min(3, len(pred_df)), "dice").index.tolist(), "Lowest segmentation Dice")

## 13. Dự đoán cho một ảnh X-quang mới

In [ ]:
def predict_file(path, max_side=CACHE_MAX_SIDE):
    with Image.open(path) as im:
        im = im.convert("RGB")
        im.thumbnail((max_side, max_side), Image.LANCZOS)
        img = torch.from_numpy(np.array(im)).permute(2, 0, 1).contiguous()
    mask, p_mal = predict(img)
    label = CLASS_NAMES[int(p_mal >= 0.5)]

    rgba = np.zeros((*mask.shape, 4))
    rgba[mask.numpy()] = [*CLASS_COLORS[int(p_mal >= 0.5)], 0.45]
    plt.figure(figsize=(6, 6))
    plt.imshow(img.permute(1, 2, 0).numpy())
    plt.imshow(rgba)
    plt.title(f"{Path(path).name}: {label} (P_malignant={p_mal:.3f})")
    plt.axis("off")
    plt.show()
    return label, p_mal, mask


example = eval_ds.samples[0][0]
predict_file(example)[:2]

## 14. Kết quả đầu ra

Các file trong `/kaggle/working` (tải về ở tab *Output* sau khi *Save Version*):

| File | Nội dung |
|---|---|
| `btxrd_instance_eomt_base_640_best.bin` | Trọng số tốt nhất (state dict), dùng với `--model.ckpt_path` |
| `checkpoints/btxrd_instance_eomt_base_640/*.ckpt` | Checkpoint Lightning (best + last, có thể resume) |
| `btxrd_instance_eomt_base_640.yaml` | Cấu hình đầy đủ của lần chạy |
| `btxrd_instance_eomt_base_640_test_predictions.csv` | Dự đoán theo ảnh trên tập test |
| `logs/` | Log CSV (và `wandb/` nếu dùng W&B) |

Đánh giá lại ở máy khác bằng CLI của dự án:
```bash
python3 main.py test -c btxrd_instance_eomt_base_640_test.yaml \
  --data.path /path/to/BTXRD \
  --model.ckpt_path btxrd_instance_eomt_base_640_best.bin
```

In [ ]:
for p in sorted(WORK_DIR.iterdir()):
    if p.name not in ("eomt", "btxrd_cache"):
        print(p.name)
# Free up the output size: the image cache can be rebuilt from the input dataset
shutil.rmtree(CACHE_DIR, ignore_errors=True)